In [231]:
import os 
import sys
from dataclasses import dataclass
sys.path.append(os.path.join(os.path.dirname("__file__"), '..'))

import numpy as np
import scipy
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

import skfem
import gmsh
import meshio
from src.utils import *

In [453]:
node, ele, _, _ = make_dom_2(h=0.8)
used, inverse_map = np.unique(ele, return_inverse=True)
if len(used) < len(node):
    node = node[used]
    ele = inverse_map.reshape(ele.shape)
mesh= Mesh(node, ele)



ValueError: operands could not be broadcast together with shapes (196,2) (196,) 

In [454]:
import numpy as np
import scipy.sparse as sp
import scipy.spatial
import matplotlib.tri as mtri
from dataclasses import dataclass
from typing import Tuple, Dict, Any

@dataclass
class Mesh:
    
    nodes: np.ndarray      # (N, 2)
    triangles: np.ndarray  # (M, 3)

    def __post_init__(self):
        self.nodes = np.asarray(self.nodes, dtype=float)
        self.triangles = np.asarray(self.triangles, dtype=int)

        
        assert len(self.nodes == len( np.unique(self.triangles)))

        self.n = len(self.nodes)
        self.m_elem = len(self.triangles)

        x = self.nodes[self.triangles]  # (M, 3, 2)
        # J[:, :, 0] = x1 - x0, J[:, :, 1] = x2 - x0
        J = np.stack((x[:, 1] - x[:, 0], x[:, 2] - x[:, 0]), axis=-1)
        det = np.linalg.det(J)
        if np.any(np.abs(det) < 1e-14): raise ValueError("Degenerate triangle")
        self.area = np.abs(det) / 2.0  # (M,)
        self.invJ = np.linalg.inv(J)   # (M, 2, 2)

        gN0 = self.invJ[:, 0, :]   # (M, 2)
        gN1 = self.invJ[:, 1, :]   # (M, 2)
        gN2 = -(gN0 + gN1) # (M, 2)
        self.grad_basis = np.stack([gN0, gN1, gN2], axis=1)  # (M, 3, 2)

        # Lumped mass vector: m_i = \int \beta_i(x) dx = \sum_{T \ni i} Area(T)/3
        self.mass = np.bincount(self.triangles.ravel(), weights=np.repeat(self.area / 3.0, 3), minlength=self.n)
        assert np.all(self.mass > 0)
        self.tri = mtri.Triangulation(self.nodes[:, 0], self.nodes[:, 1], self.triangles)
        self.find = self.tri.get_trifinder()
        
    
        self.edges = np.sort(np.concatenate([
            self.triangles[:, [0, 1]], self.triangles[:, [1, 2]], self.triangles[:, [2, 0]]
        ], axis=0), axis=1)
        unique_e, counts = np.unique(self.edges, axis=0, return_counts=True)
        self.bc_edges = unique_e[counts == 1]
        self.bc_idx = np.unique(self.bc_edges)
        self.bc = self.nodes[self.bc_idx]

    def gradient(self, values):
        """
        Lumped P1 gradient: constant gradient per simplex, scattered to nodes with area weights.
        Equivalent to vertex-centered Median Dual Green-Gauss Finite Volume gradient.
        """
        element_grads = np.einsum('ea,ead->ed', values[self.triangles], self.grad_basis)
        contrib = np.repeat(self.area[:, None] / 3.0 * element_grads, 3, axis=0)
        print(contrib.shape)
        out = np.zeros((self.n, 2))
        np.add.at(out, self.triangles.ravel(), contrib)
        return out / self.mass[:, None]

    def interpolation(self, points: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Locates containing triangles and evaluates barycentric weights for arbitrary points.
        Includes a projection fallback for points infinitesimally outside due to boundary discretization gaps.
        """
        pts = np.asarray(points, dtype=float)
        e = self.find(pts[:, 0], pts[:, 1])

        if np.any(e < 0):
            outside_idx = np.where(e < 0)[0]
            # Inward projection fallback for points along curved boundaries
            r_out = np.linalg.norm(pts[outside_idx], axis=1, keepdims=True)
            r_out = np.maximum(r_out, 1e-12)
            pts_inward = pts[outside_idx] * np.minimum(1.0, 0.995 / r_out)
            e_inward = self.find(pts_inward[:, 0], pts_inward[:, 1])
            for i, o_idx in enumerate(outside_idx):
                cand = e_inward[i]
                if cand < 0:
                    cand = 0
                e[o_idx] = cand
                pts[o_idx] = pts_inward[i]

        disp = pts - self.nodes[self.triangles[e, 0]]
        local = np.einsum('eij,ej->ei', self.invJ[e], disp)
        weights = np.column_stack((1.0 - local.sum(axis=1), local))
        weights = np.maximum(weights, 0.0)
        weights /= weights.sum(axis=1, keepdims=True)
        return self.triangles[e], weights



In [428]:
mesh = Mesh(node, ele)

In [446]:
mesh.triangles[mesh.find(0.48, 0.6), :]

array([ 56, 123, 136])